# Role-Aware SAAMR: All-Atom PE/PEAA Ionomer to OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook builds all-atom analogues of the PE/PEAA ionomer systems described by the coarse-grained paper schematic. The paper simulated a CG representation; here MuPT builds chemically explicit polyethylene/acrylic-acid-side-chain polymers with explicit sodium counterions and sends the resulting system toward OpenFF/OpenMM.

The notebook is intentionally knob-driven. It defaults to a small smoke test, while the production target is 800 chains per system.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys

import networkx as nx
import numpy as np
from rdkit import Chem


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

from mupt.builders.random_walk import random_walk_jointed_chain
from mupt.geometry.coordinates.directions import random_unit_vector
from mupt.geometry.shapes import PointCloud
from mupt.geometry.transforms.rigid import rigid_vector_coalignment
from mupt.interfaces.rdkit import primitive_from_mupt_sdf, primitive_to_rdkit_mols
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "role_aware_ionomer_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

## 1. Science Knobs

`USE_PRODUCTION_SIZE = False` keeps the notebook runnable as a demonstration. Set it to `True` to build 800 chains for the selected system.

In [ ]:
SYSTEM_SPECS = {
    "m3": {
        "pattern": ["PE", "PEAA", "PE"],
        "repeat_count": 12,
        "expected_units": 36,
    },
    "m5": {
        "pattern": ["PE", "PE", "PEAA", "PE", "PE"],
        "repeat_count": 7,
        "expected_units": 35,
    },
    "m7": {
        "pattern": ["PE", "PE", "PE", "PEAA", "PE", "PE", "PE"],
        "repeat_count": 5,
        "expected_units": 35,
    },
}

BUILD_SYSTEM_NAME = "m3"  # "m3", "m5", "m7", or "all"
USE_PRODUCTION_SIZE = False
N_CHAINS_SMOKE_TEST = 2
N_CHAINS_PRODUCTION = 800
RANDOM_SEED = 51

BACKBONE_BOND_LENGTH_A = 1.54
INTER_RESIDUE_BOND_LENGTH_A = 1.54
CHAIN_SPACING_A = 45.0
ANGLE_MAX_RAD = np.pi / 4
SODIUM_COO_DISTANCE_A = 2.4
MIN_ALLOWED_DISTANCE_A = 0.25

N_CHAINS = N_CHAINS_PRODUCTION if USE_PRODUCTION_SIZE else N_CHAINS_SMOKE_TEST
SELECTED_SYSTEMS = list(SYSTEM_SPECS) if BUILD_SYSTEM_NAME == "all" else [BUILD_SYSTEM_NAME]

print(f"Selected systems: {SELECTED_SYSTEMS}")
print(f"Chains per system: {N_CHAINS}")

## 2. Repeat Chemistry

Each `PE` or `PEAA` repeat represents a three-carbon backbone block. The acid-bearing repeat is `-CH2-CH(CH2COO-)-CH2-`. Head and tail cap residues have one linker site and explicit terminal hydrogen caps; they do not add extra carbons. Sodium is represented as a separate one-particle residue and is not covalently bonded to the polymer.

In [ ]:
REPEAT_SMILES = {
    "PE_HEAD": "[H]-[CH2:1]-[CH2]-[CH2:2]-*",
    "PE": "*-[CH2:1]-[CH2]-[CH2:2]-*",
    "PEAA": "*-[CH2:1]-[CH]([CH2][C](=O)[O-])-[CH2:2]-*",
    "PE_TAIL": "*-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "NA": "[Na+]",
}

RESNAME_MAP = {
    "PE_HEAD": "PEH",
    "PE": "PEX",
    "PEAA": "PAA",
    "PE_TAIL": "PET",
    "NA": "SOD",
}

# RDKit distance-geometry embedding can fail for these dummy-linker alkyl fragments.
# These deterministic local coordinates are enough for MuPT topology registration,
# SDF export, and a non-overlapping OpenMM starting configuration.
LOCAL_COORDS_A = {
    "PE_HEAD": {
        0: [-1.09, 0.0, 0.0],
        1: [0.0, 0.0, 0.0],
        2: [1.54, 0.0, 0.0],
        3: [3.08, 0.0, 0.0],
        5: [0.0, 1.0, 0.0],
        6: [0.0, -1.0, 0.0],
        7: [1.54, 1.0, 0.0],
        8: [1.54, -1.0, 0.0],
        9: [3.08, 1.0, 0.0],
        10: [3.08, -1.0, 0.0],
    },
    "PE": {
        1: [0.0, 0.0, 0.0],
        2: [1.54, 0.0, 0.0],
        3: [3.08, 0.0, 0.0],
        5: [0.0, 1.0, 0.0],
        6: [0.0, -1.0, 0.0],
        7: [1.54, 1.0, 0.0],
        8: [1.54, -1.0, 0.0],
        9: [3.08, 1.0, 0.0],
        10: [3.08, -1.0, 0.0],
    },
    "PEAA": {
        1: [0.0, 0.0, 0.0],
        2: [1.54, 0.0, 0.0],
        3: [1.54, 1.52, 0.0],
        4: [1.54, 2.95, 0.0],
        5: [0.55, 3.65, 0.0],
        6: [2.55, 3.65, 0.0],
        7: [3.08, 0.0, 0.0],
        9: [0.0, 1.0, 0.0],
        10: [0.0, -1.0, 0.0],
        11: [1.02, -0.72, 0.85],
        12: [3.08, 1.0, 0.0],
        13: [3.08, -1.0, 0.0],
        14: [1.05, 2.95, 0.9],
        15: [2.05, 2.95, -0.9],
    },
    "PE_TAIL": {
        1: [0.0, 0.0, 0.0],
        2: [1.54, 0.0, 0.0],
        3: [3.08, 0.0, 0.0],
        4: [4.17, 0.0, 0.0],
        5: [0.0, 1.0, 0.0],
        6: [0.0, -1.0, 0.0],
        7: [1.54, 1.0, 0.0],
        8: [1.54, -1.0, 0.0],
        9: [3.08, 1.0, 0.0],
        10: [3.08, -1.0, 0.0],
    },
    "NA": {0: [0.0, 0.0, 0.0]},
}

PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS = (5, 6)

## 3. Notebook-Local Builder

This is intentionally notebook-local. The point is to demonstrate that MuPT can already represent the role-aware hierarchy, cap residues, counterions, SDF export, and OpenMM handoff without waiting for a specialized public ionomer builder API.

In [ ]:
@dataclass(frozen=True)
class BuiltIonomerSystem:
    primitive: Primitive
    system_name: str
    chain_sequences: list[list[str]]
    sodium_count: int
    sdf_paths: list[Path]


def set_local_coordinates(primitive: Primitive, coords_by_atom_label: dict[int, list[float]]) -> None:
    """Assign deterministic local coordinates to atom children and linker connectors."""
    points = []
    for handle, atom in primitive.children_by_handle.items():
        atom_label = int(atom.label)
        position = np.array(coords_by_atom_label[atom_label], dtype=float)
        atom.shape = PointCloud(position)
        points.append(position)

    primitive.shape = PointCloud(np.vstack(points))

    for conn_handle, conn_ref in primitive.external_connectors.items():
        atom = primitive.children_by_handle[conn_ref.primitive_handle]
        atom_label = int(atom.label)
        connectors = (primitive.fetch_connector(conn_handle), primitive.fetch_connector_on_child(conn_ref))
        anchor = np.array(coords_by_atom_label[atom_label], dtype=float)
        # Map-number 1 is the incoming/head-side connector; map-number 2 is outgoing/tail-side.
        direction = -1.0 if atom.metadata.get("molAtomMapNumber") == 1 else 1.0
        linker = anchor + np.array([direction * INTER_RESIDUE_BOND_LENGTH_A, 0.0, 0.0])
        for conn in connectors:
            conn.anchor.position = anchor
            conn.linker.position = linker


def build_repeat_lexicon() -> dict[str, Primitive]:
    """Create RESIDUE -> PARTICLE primitives for repeat units and sodium."""
    lexicon = {}
    for name, smiles in REPEAT_SMILES.items():
        residue = primitive_from_smiles(
            smiles,
            label=name,
            ensure_explicit_Hs=True,
            embed_positions=False,
        )
        residue.role = PrimitiveRole.RESIDUE
        residue.metadata.update({
            "repeat_kind": name,
            "residue_name": RESNAME_MAP[name],
        })
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
            atom.metadata.setdefault("residue_name", RESNAME_MAP[name])
        set_local_coordinates(residue, LOCAL_COORDS_A[name])
        lexicon[name] = residue
    return lexicon


def expanded_sequence(system_name: str) -> list[str]:
    """Expand a system pattern and replace terminal PE blocks with cap residues."""
    spec = SYSTEM_SPECS[system_name]
    sequence = list(spec["pattern"]) * int(spec["repeat_count"])
    if len(sequence) != int(spec["expected_units"]):
        raise ValueError(f"{system_name} produced {len(sequence)} units, expected {spec['expected_units']}")
    if sequence[0] != "PE" or sequence[-1] != "PE":
        raise ValueError("Current cap logic expects chain patterns to start and end with PE")
    sequence[0] = "PE_HEAD"
    sequence[-1] = "PE_TAIL"
    return sequence


def residue_atom_position(residue: Primitive, atom_label: int) -> np.ndarray:
    """Return the centroid of the atom with the requested RDKit atom label."""
    for atom in residue.children:
        if int(atom.label) == atom_label:
            return np.array(atom.shape.centroid, dtype=float)
    raise KeyError(f"Could not find atom label {atom_label} in {residue.label}")


def set_single_atom_residue_position(residue: Primitive, position: np.ndarray) -> None:
    """Move a one-particle residue to an absolute position."""
    atom = residue.children[0]
    atom.shape = PointCloud(np.array(position, dtype=float))
    residue.shape = PointCloud(np.array(position, dtype=float))


def place_sodium_near_peaa(peaa_residue: Primitive, rng: np.random.Generator) -> np.ndarray:
    """Return a sodium position near the carboxylate oxygen midpoint."""
    oxy_1 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[0])
    oxy_2 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[1])
    midpoint = 0.5 * (oxy_1 + oxy_2)
    normal = np.cross(oxy_2 - oxy_1, np.array([1.0, 0.0, 0.0]))
    if np.linalg.norm(normal) < 1.0e-8:
        normal = random_unit_vector()
    normal = normal / np.linalg.norm(normal)
    jitter = 0.15 * random_unit_vector()
    direction = normal + jitter
    direction = direction / np.linalg.norm(direction)
    return midpoint + SODIUM_COO_DISTANCE_A * direction


def residue_backbone_span(residue: Primitive) -> tuple[np.ndarray, np.ndarray]:
    """Return local map-1 and map-2 atom positions for repeat placement."""
    by_map_number = {
        int(atom.metadata["molAtomMapNumber"]): np.array(atom.shape.centroid, dtype=float)
        for atom in residue.children
        if "molAtomMapNumber" in atom.metadata
    }
    return by_map_number[1], by_map_number[2]


def place_segment_by_random_walk(segment: Primitive, residue_handles: list, chain_idx: int) -> None:
    """Place repeat residues along a random walk without mutating connector geometry."""
    step_count = len(residue_handles)
    start = np.array([chain_idx * CHAIN_SPACING_A, 0.0, 0.0])
    walk_points = list(random_walk_jointed_chain(
        step_size=3.0 * BACKBONE_BOND_LENGTH_A + INTER_RESIDUE_BOND_LENGTH_A,
        n_steps_max=step_count,
        initial_point=start,
        initial_direction=random_unit_vector(),
        clip_angle=ANGLE_MAX_RAD,
        dimension=3,
    ))
    for handle, step_start, step_end in zip(residue_handles, walk_points[:-1], walk_points[1:]):
        residue = segment.children_by_handle[handle]
        local_start, local_end = residue_backbone_span(residue)
        transform = rigid_vector_coalignment(local_start, local_end, step_start, step_end, t1=0.0, t2=0.0)
        residue.rigidly_transform(transform)


def build_ionomer_system(system_name: str, n_chains: int, random_seed: int) -> BuiltIonomerSystem:
    """Build one all-atom PE/PEAA ionomer system with explicit sodium counterions."""
    rng = np.random.default_rng(random_seed)
    lexicon = build_repeat_lexicon()
    sequence_template = expanded_sequence(system_name)

    universe = Primitive(label=f"ionomer_{system_name}", role=PrimitiveRole.UNIVERSE)
    universe.metadata.update({
        "system_name": system_name,
        "n_chains": str(n_chains),
        "terminal_caps": "explicit_hydrogen_residues",
    })

    chain_sequences = []
    sodium_count = 0

    for chain_idx in range(n_chains):
        segment = Primitive(label=f"chain_{chain_idx:04d}", role=PrimitiveRole.SEGMENT)
        segment.metadata.update({"system_name": system_name, "chain_index": str(chain_idx)})
        residue_handles = []
        chain_sequences.append(sequence_template)

        for repeat_idx, repeat_kind in enumerate(sequence_template):
            residue = lexicon[repeat_kind].copy()
            residue.label = f"repeat_{repeat_idx:03d}_{repeat_kind}"
            residue.role = PrimitiveRole.RESIDUE
            residue.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "repeat_kind": repeat_kind,
                "residue_name": RESNAME_MAP[repeat_kind],
            })
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update({
                    "system_name": system_name,
                    "chain_index": str(chain_idx),
                    "repeat_index": str(repeat_idx),
                    "repeat_kind": repeat_kind,
                    "residue_name": RESNAME_MAP[repeat_kind],
                })
            residue_handles.append(segment.attach_child(residue))

        segment.set_topology(
            nx.path_graph(residue_handles, create_using=TopologicalStructure),
            max_registration_iter=100,
        )

        place_segment_by_random_walk(segment, residue_handles, chain_idx)

        universe.attach_child(segment)

        for repeat_idx, repeat_kind in enumerate(sequence_template):
            if repeat_kind != "PEAA":
                continue
            peaa_residue = segment.children[repeat_idx]
            sodium = lexicon["NA"].copy()
            sodium.label = f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}"
            sodium.role = PrimitiveRole.RESIDUE
            sodium_position = place_sodium_near_peaa(peaa_residue, rng)
            set_single_atom_residue_position(sodium, sodium_position)
            sodium.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "associated_peaa_residue": peaa_residue.label,
                "residue_name": RESNAME_MAP["NA"],
            })
            for atom in sodium.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update(sodium.metadata)

            sodium_segment = Primitive(
                label=f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}",
                role=PrimitiveRole.SEGMENT,
            )
            sodium_segment.metadata.update(sodium.metadata)
            sodium_segment.attach_child(sodium)
            universe.attach_child(sodium_segment)
            sodium_count += 1

    return BuiltIonomerSystem(
        primitive=universe,
        system_name=system_name,
        chain_sequences=chain_sequences,
        sodium_count=sodium_count,
        sdf_paths=[],
    )

## 4. Build Systems

For production-size builds, run one system at a time unless you know your workstation has enough memory.

In [ ]:
built_systems = []
for system_name in SELECTED_SYSTEMS:
    built = build_ionomer_system(system_name, n_chains=N_CHAINS, random_seed=RANDOM_SEED)
    built_systems.append(built)

    sequence = expanded_sequence(system_name)
    print(f"{system_name}: chains={N_CHAINS}")
    print(f"  repeat units per chain: {len(sequence)}")
    print(f"  PEAA per chain: {sequence.count('PEAA')}")
    print(f"  sodium ions: {built.sodium_count}")
    print(f"  total particles: {len(built.primitive.leaves)}")

## 5. Coordinate Diagnostics

The starting coordinates are not a melt-packing algorithm. They are non-overlapping random-walk coordinates intended for OpenMM minimization and vacuum collapse before optional periodic NPT.

In [ ]:
def primitive_positions(primitive: Primitive) -> np.ndarray:
    """Collect leaf-particle coordinates from a MuPT primitive."""
    return np.vstack([np.array(leaf.shape.centroid, dtype=float) for leaf in primitive.leaves])


def minimum_pair_distance(positions: np.ndarray) -> float:
    """Return nearest-neighbor distance without allocating an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions).query(positions, k=2)
    return float(np.min(distances[:, 1]))


for built in built_systems:
    positions = primitive_positions(built.primitive)
    span = positions.max(axis=0) - positions.min(axis=0)
    min_distance = minimum_pair_distance(positions)
    print(f"{built.system_name}: minimum atom distance = {min_distance:.3f} A")
    print(f"{built.system_name}: coordinate span = {span[0]:.1f} x {span[1]:.1f} x {span[2]:.1f} A")
    if min_distance < MIN_ALLOWED_DISTANCE_A:
        raise ValueError(
            f"{built.system_name} has an atom-atom distance below {MIN_ALLOWED_DISTANCE_A} A. "
            "Increase CHAIN_SPACING_A or adjust local fragment coordinates."
        )

## 6. Export Role-Aware SDF Files

Each polymer chain and sodium ion is exported as an independent SDF molecule record. Sodium residues remain associated with their PEAA repeat through metadata, not covalent bonds.

In [ ]:
def write_system_sdfs(built: BuiltIonomerSystem) -> list[Path]:
    """Write one SDF file per exported molecule segment."""
    sdf_dir = OUTPUT_ROOT / built.system_name / "sdf"
    sdf_dir.mkdir(parents=True, exist_ok=True)
    sdf_paths = []

    resname_map = {
        residue.label: residue.metadata.get("residue_name", "UNK")
        for segment in built.primitive.children
        for residue in segment.children
    }
    mols = primitive_to_rdkit_mols(built.primitive, resname_map=resname_map)
    for mol_idx, mol in enumerate(mols):
        label = mol.GetProp("mupt_segment_label") if mol.HasProp("mupt_segment_label") else f"mol_{mol_idx:05d}"
        path = sdf_dir / f"{built.system_name}_{label}.sdf"
        writer = Chem.SDWriter(str(path))
        writer.write(mol)
        writer.close()
        sdf_paths.append(path)

    return sdf_paths


for idx, built in enumerate(built_systems):
    sdf_paths = write_system_sdfs(built)
    built_systems[idx] = BuiltIonomerSystem(
        primitive=built.primitive,
        system_name=built.system_name,
        chain_sequences=built.chain_sequences,
        sodium_count=built.sodium_count,
        sdf_paths=sdf_paths,
    )
    print(f"{built.system_name}: wrote {len(sdf_paths)} SDF file(s) to {sdf_paths[0].parent}")

## 7. Fast MuPT SDF Validation

This reload uses MuPT metadata only. It avoids expensive bond and shape reconstruction, which matters for production-size SDF sets.

In [ ]:
for built in built_systems:
    reconstructed = primitive_from_mupt_sdf(
        built.sdf_paths,
        reconstruct_bonds=False,
        reconstruct_shapes=False,
    )
    chain_segments = [segment for segment in reconstructed.children if str(segment.label).startswith("chain_")]
    sodium_segments = [segment for segment in reconstructed.children if str(segment.label).startswith("sodium_")]
    peaa_residues = [
        residue
        for segment in chain_segments
        for residue in segment.children
        if str(residue.label).endswith("_PEAA")
    ]

    assert len(chain_segments) == N_CHAINS
    assert len(sodium_segments) == built.sodium_count
    assert len(peaa_residues) == built.sodium_count
    print(
        f"{built.system_name}: reconstructed chains={len(chain_segments)}, "
        f"PEAA={len(peaa_residues)}, sodium={len(sodium_segments)}"
    )

## 8. OpenFF/OpenMM Setup

The cells below mirror the companion OpenFF/OpenMM notebook: use OpenFF NAGL GNN charges for polymer chains, avoid AM1-BCC fallback, minimize and briefly collapse in vacuum, then optionally wrap a padded periodic box for NPT. Keep `RUN_OPENFF_PARAMETERIZATION = False` until the smoke-test SDFs look right.

In [ ]:
try:
    from functools import reduce

    from openff.interchange import Interchange
    from openff.toolkit import ForceField, Molecule
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError("OpenFF NAGL backend is unavailable; install openff-nagl")
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"
RUN_OPENFF_PARAMETERIZATION = False
RUN_VACUUM_COLLAPSE = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_PERIODIC_NPT = False
VACUUM_COLLAPSE_STEPS = 500
PERIODIC_NPT_STEPS = 250
PERIODIC_PADDING_NM = 3.0

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")

In [ ]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol) -> None:
    """Copy SDF atom-property metadata into OpenFF atom metadata."""
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        off_atom.metadata.update({
            "residue_name": str(props.get("residue_name", props.get("mupt_residue_label", "UNK"))),
            "residue_number": str(props.get("mupt_residue_index", "1")),
            "chain_id": str(props.get("chain_id", "A")),
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


def load_rdkit_sdf(path: Path) -> Chem.Mol:
    """Load one SDF molecule without stripping explicit hydrogens."""
    supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
    mol = supplier[0]
    if mol is None:
        raise ValueError(f"Could not read {path}")
    Chem.SanitizeMol(Chem.Mol(mol))
    return mol


openff_molecules_by_system = {}
if OPENFF_AVAILABLE:
    for built in built_systems:
        off_molecules = []
        for path in built.sdf_paths:
            rdkit_mol = load_rdkit_sdf(path)
            off_mol = Molecule.from_rdkit(
                rdkit_mol,
                allow_undefined_stereo=True,
                hydrogens_are_explicit=True,
            )
            transfer_rdkit_metadata_to_openff(rdkit_mol, off_mol)
            off_molecules.append(off_mol)
        openff_molecules_by_system[built.system_name] = off_molecules
        print(f"{built.system_name}: created {len(off_molecules)} OpenFF molecule(s)")
else:
    print("Skipping OpenFF molecule conversion because openff-toolkit is unavailable.")

In [ ]:
interchanges_by_system = {}

if OPENFF_AVAILABLE and NAGL_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for built in built_systems:
        mol_interchanges = []
        off_molecules = openff_molecules_by_system[built.system_name]
        for mol_idx, off_mol in enumerate(off_molecules):
            is_sodium = off_mol.n_atoms == 1 and off_mol.atom(0).symbol == "Na"
            print(f"{built.system_name}: parameterizing molecule {mol_idx + 1}/{len(off_molecules)}")
            if not is_sodium:
                off_mol.assign_partial_charges(
                    partial_charge_method=PARTIAL_CHARGE_METHOD,
                    toolkit_registry=nagl_registry,
                )
            else:
                off_mol.partial_charges = [1.0] * off_unit.elementary_charge

            mol_inc = ff.create_interchange(
                off_mol.to_topology(),
                charge_from_molecules=[off_mol],
            )
            mol_inc.box = None
            mol_interchanges.append(mol_inc)

        interchange = reduce(Interchange.combine, mol_interchanges)
        interchange.box = None
        interchanges_by_system[built.system_name] = interchange
        print(f"{built.system_name}: combined vacuum interchange with {interchange.topology.n_atoms} atoms")
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print("Parameterization skipped because OpenFF NAGL is unavailable; no AM1-BCC fallback is used.")
else:
    print("Parameterization skipped. Set RUN_OPENFF_PARAMETERIZATION = True after checking the smoke-test SDFs.")

## 9. Vacuum Collapse and Optional Periodic NPT

This follows the companion notebook's pragmatic route: collapse in vacuum first to avoid building an enormous sparse PME grid around random-walk starting coordinates. Only enable periodic NPT after inspecting the vacuum result.

In [ ]:
def set_periodic_box_around_positions(interchange, padding_nm: float):
    """Translate coordinates into a padded orthorhombic periodic box."""
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    mins = positions_nm.min(axis=0)
    maxs = positions_nm.max(axis=0)
    shifted_positions = positions_nm - mins + padding_nm
    box_lengths = (maxs - mins) + 2 * padding_nm
    interchange.positions = shifted_positions * off_unit.nanometer
    interchange.box = np.diag(box_lengths) * off_unit.nanometer
    return box_lengths


if interchanges_by_system and OPENMM_AVAILABLE and RUN_VACUUM_COLLAPSE:
    for built in built_systems:
        interchange = interchanges_by_system[built.system_name]
        openmm_dir = OUTPUT_ROOT / built.system_name / "OpenMM"
        openmm_dir.mkdir(parents=True, exist_ok=True)

        temperature = 300.0 * omm_unit.kelvin
        pressure = 1.0 * omm_unit.atmosphere
        vacuum_time_step = 0.5 * omm_unit.femtosecond
        periodic_time_step = 1.0 * omm_unit.femtosecond
        friction = 1.0 / omm_unit.picosecond

        print(f"{built.system_name}: creating non-periodic vacuum OpenMM simulation")
        vacuum_integrator = LangevinMiddleIntegrator(temperature, friction, vacuum_time_step)
        vacuum_simulation = interchange.to_openmm_simulation(
            integrator=vacuum_integrator,
            combine_nonbonded_forces=False,
        )
        vacuum_simulation.minimizeEnergy()
        if VACUUM_COLLAPSE_STEPS > 0:
            vacuum_simulation.reporters.append(
                StateDataReporter(sys.stdout, max(1, VACUUM_COLLAPSE_STEPS // 10), step=True, potentialEnergy=True, temperature=True)
            )
            vacuum_simulation.step(VACUUM_COLLAPSE_STEPS)

        vacuum_state = vacuum_simulation.context.getState(getPositions=True, getEnergy=True)
        collapsed_positions_nm = vacuum_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
        interchange.positions = collapsed_positions_nm * off_unit.nanometer

        with open(openmm_dir / "vacuum_system.xml", "w") as handle:
            handle.write(XmlSerializer.serialize(vacuum_simulation.system))
        with open(openmm_dir / "vacuum_integrator.xml", "w") as handle:
            handle.write(XmlSerializer.serialize(vacuum_integrator))
        with open(openmm_dir / "vacuum_state.xml", "w") as handle:
            handle.write(XmlSerializer.serialize(vacuum_state))
        with open(openmm_dir / "vacuum_topology.pdb", "w") as handle:
            PDBFile.writeFile(vacuum_simulation.topology, vacuum_state.getPositions(), handle)

        if RUN_PERIODIC_NPT:
            box_lengths = set_periodic_box_around_positions(interchange, PERIODIC_PADDING_NM)
            periodic_integrator = LangevinMiddleIntegrator(temperature, friction, periodic_time_step)
            periodic_system = interchange.to_openmm(combine_nonbonded_forces=False)
            periodic_system.addForce(MonteCarloBarostat(pressure, temperature))
            periodic_simulation = openmm.app.Simulation(interchange.to_openmm_topology(), periodic_system, periodic_integrator)
            periodic_simulation.context.setPositions(interchange.positions.m_as(off_unit.nanometer) * omm_unit.nanometer)
            periodic_simulation.minimizeEnergy()
            if PERIODIC_NPT_STEPS > 0:
                periodic_simulation.step(PERIODIC_NPT_STEPS)
            print(f"{built.system_name}: periodic box lengths after padding: {box_lengths}")
else:
    print("OpenMM run skipped because no Interchange was created or RUN_VACUUM_COLLAPSE is False.")

## 10. Notes for Production Runs

For the production target, set `USE_PRODUCTION_SIZE = True` and run one system at a time. The expected sodium counts are 9,600 for `m3`, 5,600 for `m5`, and 4,000 for `m7`. Keep the vacuum-collapse-first workflow unless you already have dense initial coordinates; otherwise the periodic PME grid can be dominated by empty space.